# Deep Learning: Methodological Depth & Comparative Analysis

This notebook is part of the Deep Learning course at Istinye University, focusing on the comparative analysis of MLP models, activation functions, regularization techniques, and optimization methods.

## 1. Data Preparation

Perform data normalization and batching.

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Device configuration (Support for Mac MPS, NVIDIA CUDA, or CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

# 1.1. Preprocessing: Normalize to [-1, 1] for training stability (Week 4)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

# 1.2. Dataset Loading (Experience - E)
train_full = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# 1.3. Validation Strategy: 20% split to monitor generalization (Week 2)
train_data, val_data = random_split(train_full, [50000, 10000])

# 1.4. DataLoaders (Ensuring i.i.d. with shuffle=True for training)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

print(f"PyTorch Version: {torch.__version__}")
print(f"Using Device: {device}")
print(f"Dataset Sizes -> Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_set)}")

PyTorch Version: 2.11.0
Using Device: mps
Dataset Sizes -> Train: 50000 | Val: 10000 | Test: 10000


## 2. Model Architecture & Activation Analysis

We use different activation functions and regularization techniques to compare results.

In [5]:
import torch.nn as nn

# Deep MLP Architecture: 5-layer depth to analyze gradient flow (Week 3)
class DeepMLP(nn.Module):
    def __init__(self, activation_fn):
        super(DeepMLP, self).__init__()
        self.flatten = nn.Flatten()
        
        # Layer stack: 784 -> 512 -> 256 -> 128 -> 64 -> 10
        self.layers = nn.ModuleList([
            nn.Linear(784, 512),
            nn.Linear(512, 256),
            nn.Linear(256, 128),
            nn.Linear(128, 64),
            nn.Linear(64, 10)
        ])
        self.activation = activation_fn

    def forward(self, x):
        x = self.flatten(x)
        # Apply activation to all layers except the output layer
        for i in range(len(self.layers) - 1):
            x = self.activation(self.layers[i](x))
        x = self.layers[-1](x)
        return x

In [6]:
# Function to measure gradient magnitude at the input layer
def analyze_gradient_flow(activation_name, activation_fn):
    # Initialize model with the specific activation function
    model = DeepMLP(activation_fn).to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Fetch a single mini-batch from the training set
    images, labels = next(iter(train_loader))
    images, labels = images.to(device), labels.to(device)
    
    # Perform Forward and Backward Pass (Backpropagation)
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    
    # Extract gradient norm of the first layer (furthest from the output)
    grad_norm = model.layers[0].weight.grad.norm().item()
    print(f"{activation_name.upper()} -> Input Layer Gradient Norm: {grad_norm:.10f}")

print("--- Vanishing Gradient Experiment ---")
analyze_gradient_flow("sigmoid", nn.Sigmoid())
analyze_gradient_flow("relu", nn.ReLU())

--- Vanishing Gradient Experiment ---
SIGMOID -> Input Layer Gradient Norm: 0.0015892168
RELU -> Input Layer Gradient Norm: 0.1118951961


## 3. Training Function and Methodology

This section includes functions for training and monitoring performance.

In [ ]:
def get_gradient_norms(model):
    """
    Her katmandaki gradyan büyüklüklerini (L2 norm) hesaplar.
    Bu, Vanishing Gradient problemini görselleştirmek için kritiktir.
    """
    grads = []
    for name, param in model.named_parameters():
        # Sadece ağırlık (weight) gradyanlarını alıyoruz, bias'ları geçebiliriz
        if param.grad is not None and 'weight' in name:
            grads.append(param.grad.norm().item())
    return grads
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    train_losses = []
    # Backpropagation allows updating our weights via gradient descent (e.g., SGD/Adam).
    # Loss function calculates predicted error, which is distributed backward (backprop).
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Gradient Flow: Calculation of gradients via loss using the Autograd mechanism
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")
        
    return train_losses

## 4. Visualization (Learning Curves)

Function to visualize training loss curves.

In [ ]:
def plot_learning_curves(losses, title="Learning Curves"):
    plt.figure(figsize=(10, 6))
    plt.plot(losses, label='Training Loss', color='royalblue', linewidth=2)
    plt.title(title, fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

## 5. Experiment Execution

Run different scenarios and compare the results.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLPModel(activation='relu', use_dropout=True, use_bn=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) # Weight Decay = L2 Regularization

losses = train_model(model, train_loader, criterion, optimizer, epochs=5)
plot_learning_curves(losses, "ReLU with Dropout & Batch Norm")

In [ ]:
# Deney: Sigmoid (Vanishing Gradient riski) vs ReLU
activations = ['sigmoid', 'relu']
results = {}

for act in activations:
    print(f"\n--- Testing with {act.upper()} ---")
    # Batch Norm kapalı (Gradyan sorununu daha net görmek için)
    model = MLPModel(activation=act, use_dropout=False, use_bn=False).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01) # Daha 'noisy' ve geleneksel olan SGD
    
    losses = train_model(model, train_loader, criterion, optimizer, epochs=5)
    results[act] = losses

# Kıyaslama Raporu için Notlar (REPORT.md):
# 1. Sigmoid: Doyma (Saturation) nedeniyle gradyanlar 0'a yaklaşır, eğitim yavaşlar[cite: 824].
# 2. ReLU: z > 0 olduğu sürece gradyan 1'dir, Vanishing Gradient problemini çözer[cite: 796, 1132].


--- Testing with SIGMOID ---


NameError: name 'MLPModel' is not defined